# Model debug: why did TPGSR not help PaddleOCR?

**Where we are (Lecture 8: data -> model -> *debug* -> deploy).** `tpgsr_whiteboard_finetune.ipynb` failed its rule on the unseen board B3: CER 0.166 vs raw 0.153, and fine-tuning never beat the pretrained model (it kept epoch 0). We know *that* it failed. This notebook asks *why*, with forward passes only (no training, ~5 min on a Mac).

**How TPGSR is built** (`models/TPGSR/model/tsrn.py`, `TSRN_TL`), in plain words:
1. `block1`: a 9 x 9 convolution turns the 16 x 64 word into 64 feature maps.
2. `block2`-`block6`: five recurrent blocks. Each one runs two 3 x 3 convolutions, then a **vertical GRU** (where the text prior is joined in), then a **horizontal GRU across the whole width**. A GRU passes information along its whole row or column, so *in theory* every pixel sees the whole word after one block.
3. **The text prior**: a frozen CRNN reads the word and gives, for 26 time steps from left to right, a probability for each of 37 characters (blank, 0-9, a-z). `InfoGen` turns that into 32 feature maps and stretches them across the 64 columns. It carries **no vertical information**: it only says *"around this x, the letter is probably an 'e'"*.
4. `block7`-`block8`: a convolution, then a 2x pixel-shuffle upsampler gives the 32 x 128 output.

**Four questions, one part each:**

| Part | Question | Test | Lecture 8 idea |
|---|---|---|---|
| A | Is the text prior the bottleneck? | same words, 5 different priors, including the **true label** | counterfactual: change one input, watch the output |
| B | Does each output pixel really see the whole word? | blank a 4 px strip, measure how far the output changes | occlusion; effective vs theoretical receptive field |
| C | What happens inside, block by block? | feature maps, the prior, and where the change goes | activation maps (as in `sr_debug_activations.ipynb`) |
| D | Where do the pink tint and visible box edges come from, and do they matter? | colour shift on board pixels; CER with the shift removed | sanity check of a side effect |

All tests restore every word under 32 px (we debug the **network**, not the size gate). Main set: **B3** (20 lines whose label has one token per word box, ~54 words). Second opinion: the **synthetic validation** lines.

In [1]:
import sys
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "scripts").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")

REPO = find_repo_root()
sys.path.insert(0, str(REPO / "scripts"))
import tpgsr_debug as dbg
pd.set_option("display.width", 200)
ctx = dbg.setup()        # loads the kept model, the word pairs and the labelled B3 / synthetic-val words

No Transformation module specified
No SequenceModeling module specified
model: models/superres/tpgsr_whiteboard_best.pt (epoch 0) on mps
B3: 20 lines with one label token per word box, 54 words (54 under 32 px)
synthetic val: 16 lines with one label token per word box, 44 words (40 under 32 px)


## Part A. Is the text prior the bottleneck?

The **same** words go through the **same** network five times; only the prior changes:

| Prior | What it is | Why it is in the test |
|---|---|---|
| CRNN on far word (normal) | what TPGSR gets in use | the baseline |
| CRNN on clear word | the CRNN reads the clear photo | the best *this* CRNN could give |
| **true label** | one-hot letters of the real word, spread evenly over the 26 steps | a *perfect* prior (an oracle) |
| wrong label | the label of another word | counterfactual: can the prior *steer* the drawing? |
| no letters (all blank) | "nothing here" | what the network does without letter knowledge |

Columns: `word_cer` and `exact` = PaddleOCR on the restored word alone; `line_cer` = whole lines with these words pasted back; `pixel_error` = mean difference to the clear word (grey levels, 0-255); `moved_vs_normal` = how much the picture changes compared with the normal prior (grey levels).

**How to read it**
- **True label much better than normal** (lower CER): the prior is the bottleneck. A better prior (e.g. a CRNN fine-tuned on our handwriting) could rescue TPGSR.
- **All five about the same, and `moved_vs_normal` under ~1 grey level**: the network **ignores its prior**. Then the prior was never the problem, and the generator itself cannot draw letters it cannot see.
- **Wrong label visibly worse**: the prior does steer the drawing, so a wrong prior is harmful.

In [2]:
a_table, a_words = dbg.part_a_prior(ctx)
display(a_table.round(3))
display(Image(filename=str(dbg.OUT / "a_prior_examples.png")))

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


,set,version,word_cer,exact,pixel_error,moved_vs_normal,line_cer
0,B3,far word as photographed,0.246,0.204,NaN,NaN,0.165
1,B3,"16px path, no network",0.295,0.148,NaN,NaN,0.189
2,B3,clear photo,0.062,0.741,NaN,NaN,0.067
3,B3,CRNN on far word (normal),0.325,0.167,22.352,0.000,0.274
4,B3,CRNN on clear word,0.322,0.148,22.405,0.615,0.265
5,B3,true label,0.317,0.148,22.435,0.986,0.260
6,B3,wrong label,0.329,0.148,22.483,1.088,0.273
7,B3,no letters (all blank),0.323,0.167,22.440,0.613,0.267
8,synthetic val,far word as photographed,0.150,0.725,NaN,NaN,0.210
9,synthetic val,"16px path, no network",0.228,0.500,NaN,NaN,0.199


<IPython.core.display.Image object>

## Part B. Does each output pixel really see the whole word?

**Occlusion.** We blank a 4 px wide strip of the 16 x 64 input (fill it with the board colour) and measure how much the output changes **at every distance from the strip**. We slide the strip across the word, over ~40 B3 words.

Two ways, because the prior is *also* computed from the input:
- **network only (prior fixed)**: the prior from the unblanked word, so only the convolutions and GRUs can carry the change. This is the network's own receptive field.
- **as used (prior recomputed)**: the CRNN also sees the blanked word, as in real use.
- **bicubic, no network**: the reference for "purely local": bicubic only mixes the nearest 2 px.

**How to read it.** A letter is ~8 px wide in the input, the word 64 px.
- `90% of the influence within X px`: if X is about a letter or less, the network behaves **locally**, like our old CNNs, whatever the GRUs could do in theory. (Lecture 8: the *effective* receptive field is usually much smaller than the theoretical one.)
- `beyond one letter`: the share of the influence that reaches past 8 px. Bicubic is 0%. A whole-word network should show a clear share here.
- The change maps: a thin bright diagonal = local; bright full rows = global.

In [3]:
b_summary, b_profile = dbg.part_b_reach(ctx)
for mode, r in b_summary.items():
    print(f"{mode:<28} 90% of the influence within {r['radius_90']:>2} px | {r['beyond_one_letter']:.0%} beyond one letter "
          f"| change at the strip {r['change_at_strip']:.2f} grey levels")
display(Image(filename=str(dbg.OUT / "b_reach.png")))

network only (prior fixed)   90% of the influence within 31 px | 26% beyond one letter | change at the strip 20.13 grey levels
as used (prior recomputed)   90% of the influence within 31 px | 28% beyond one letter | change at the strip 20.13 grey levels
bicubic, no network          90% of the influence within  2 px | 0% beyond one letter | change at the strip 22.46 grey levels


<IPython.core.display.Image object>

## Part C. Inside the network, block by block

First a **sanity check** (Lecture 8: check your debugging tools): the step-by-step trace below is TPGSR's `forward()` written out by hand, so it must give exactly the same output as calling the model. If it prints MISMATCH, ignore this part.

1. **One word, layer by layer** (`c_layers.png`): input, the prior map (from the CRNN and from the true label), the mean activation of every block, the output, and the change compared with the 16px path (blue = darker, red = lighter; black outline = the strokes of the clear word).
2. **The prior itself** (`c_prior_matrix.png`): 37 characters x 26 time steps. Dark = likely. Compare the CRNN's guess with the true label.
3. **How much the true label changes each block** (all B3 words): the size of the difference, relative to the block's own size. Values near 0 at every block mean the prior hardly reaches the features.
4. **Where the network changes the picture** (all B3 words): split into the clear word's strokes, the 2 px next to them, and the board. `mean_change` < 0 = darker. `error_*` = difference to the clear word: if TPGSR's error is lower than the 16px path's *on the strokes*, it is rebuilding letters; if only on the board, it is just cleaning the background (the same shortcut as our old CNNs).

To look at another word, pass part of its label, e.g. `dbg.part_c_inside(ctx, word="occlusion")`.

In [4]:
c_effect, c_where, c_pick = dbg.part_c_inside(ctx)
print("example word:", c_pick["label"], f"({c_pick['native_h']:.0f} px tall in the far photo)")
display(c_effect.round(3))
display(c_where.round(2))
display(Image(filename=str(dbg.OUT / "c_layers.png")))
display(Image(filename=str(dbg.OUT / "c_prior_matrix.png")))

sanity check: step-by-step trace vs calling the model: largest difference 0.00e+00 (OK)
example word: saliency (9 px tall in the far photo)


,block,prior_effect
0,block1,0.000
1,block2,0.051
2,block3,0.058
3,block4,0.053
4,block5,0.033
5,block6,0.014
6,block7,0.027


,band,share_of_pixels,mean_change,error_16px_path,error_tpgsr
0,stroke,0.15,1.29,48.72,49.37
1,near stroke (2 px),0.27,3.82,22.77,21.91
2,board,0.58,0.45,16.31,15.66


<IPython.core.display.Image object>

<IPython.core.display.Image object>

## Part D. The pink tint and the visible box edges

In `visual_comparison.png` the restored words were slightly pink, so their box edges showed. We measure the colour shift of TPGSR's output against the 16px path **on board pixels only** (no strokes), per channel, on B3, the synthetic validation words and 200 TextZoom words (the data TPGSR learned on: if the shift is large there too, it was learned, not caused by our boards).

Then we remove each word's own shift and read the lines again. **If the CER barely moves, the tint is ugly but not why TPGSR fails.**

In [5]:
d_colour, d_cer = dbg.part_d_colour(ctx)
display(d_colour.round(2))
display(d_cer.round(3))

,set,channel,board_shift,board_level_16px
0,B3,R,1.31,169.81
1,B3,G,-0.11,172.80
2,B3,B,0.14,168.91
3,synthetic val,R,-1.24,183.07
4,synthetic val,G,-2.87,183.54
5,synthetic val,B,-1.78,178.53
6,TextZoom (training data),R,-1.44,165.38
7,TextZoom (training data),G,-2.51,157.40
8,TextZoom (training data),B,-3.13,143.72


,set,version,line_cer
0,B3,TPGSR as is,0.274
1,B3,"TPGSR, board colour matched",0.271
2,synthetic val,TPGSR as is,0.248
3,synthetic val,"TPGSR, board colour matched",0.224


## Summary file and how to put it together

In [6]:
print(dbg.summary_text(a_table, b_summary, c_effect, c_where, c_pick, d_colour, d_cer))
print("saved in", dbg.OUT.relative_to(REPO))

=== A. Is the text prior the bottleneck? (words < 32 px; CER lower = better) ===
          set                   version  word_cer  exact  pixel_error  moved_vs_normal  line_cer
           B3  far word as photographed     0.246  0.204          NaN              NaN     0.165
           B3     16px path, no network     0.295  0.148          NaN              NaN     0.189
           B3               clear photo     0.062  0.741          NaN              NaN     0.067
           B3 CRNN on far word (normal)     0.325  0.167       22.352            0.000     0.274
           B3        CRNN on clear word     0.322  0.148       22.405            0.615     0.265
           B3                true label     0.317  0.148       22.435            0.986     0.260
           B3               wrong label     0.329  0.148       22.483            1.088     0.273
           B3    no letters (all blank)     0.323  0.167       22.440            0.613     0.267
synthetic val  far word as photographed     0.

**Putting the parts together.** The most likely outcomes and what each means for the write-up:

| A (prior) | B (reach) | Conclusion |
|---|---|---|
| true label helps a lot | wide | The network *can* use letter knowledge; our CRNN just does not have it for handwriting. Next step would be a handwriting-trained prior. |
| nothing changes the output | narrow (about a letter) | TPGSR on our words behaves like a **local** filter that ignores its prior: the same failure as our own CNNs, despite the GRUs. |
| nothing changes the output | wide | It sees the whole word but has no use for the letter information: the generator learned TextZoom's printed shapes, not handwriting. |
| wrong label hurts, true label does not help | any | The prior steers the drawing, but even perfect letters do not give readable strokes at this size. |

Write the result in the report whichever row it is: a negative result **with a measured reason** is what the debug step is for.